# Demand Prophet — HGT-TFT Training Notebook

Colab/Kaggle-ready training entry point for the Demand Prophet agent.

**Inputs**: `data/<city>/demand_features.parquet`, `weather_features.parquet`, `store_features.parquet`
**Outputs**: MLflow run with `convergence_speedup`, `mumbai_calibration_coverage_90` metrics; HGT-TFT checkpoint registered in MLflow Model Registry as `demand_prophet_hgt_tft` (Staging → Production via runbook).

Honors invariants I-1 ($0 cost), I-2 (independent reward), I-3 (deterministic schema), I-13 (deterministic serialization).

In [ ]:
# 1. Install dependencies (Colab/Kaggle only — local uses pyproject.toml)
%pip install --quiet torch torch-geometric pytorch-forecasting mlflow mapie pyro-ppl

In [ ]:
# 2. Imports + reproducibility seeds
from __future__ import annotations
import os, random, numpy as np, torch, mlflow, pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={DEVICE}, torch={torch.__version__}')

In [ ]:
# 3. MLflow tracking — set MLFLOW_TRACKING_URI env var
mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000'))
CITY = os.environ.get('CITY', 'bengaluru')
EXPERIMENT = f'{CITY}_demand_prophet' if CITY == 'bengaluru' else f'mumbai_transfer_demand_prophet'
mlflow.set_experiment(EXPERIMENT)
print(f'experiment={EXPERIMENT}')

In [ ]:
# 4. Load Feast offline tables
from pathlib import Path
data_dir = Path('data') / CITY
demand = pd.read_parquet(data_dir / 'demand_features.parquet')
weather = pd.read_parquet(data_dir / 'weather_features.parquet')
stores = pd.read_parquet(data_dir / 'store_features.parquet')
print(f'demand={len(demand)}, weather={len(weather)}, stores={len(stores)}')

In [ ]:
# 5. Build HGT-TFT model (uses agents/demand_prophet/models/hybrid.py)
import sys; sys.path.append('.')
from agents.demand_prophet.models.hybrid import build_hybrid_model
model = build_hybrid_model(num_stores=len(stores['store_id'].unique()), num_skus=demand['sku_id'].nunique())
print(model)

In [ ]:
# 6. Train one epoch with MLflow logging
from agents.demand_prophet.training.train import train_one_epoch
with mlflow.start_run() as run:
    metrics = train_one_epoch(model, demand, weather, stores, device=DEVICE, seed=SEED)
    mlflow.log_metrics(metrics)
    mlflow.log_params({'seed': SEED, 'city': CITY, 'device': str(DEVICE)})
    print(metrics)

In [ ]:
# 7. Conformal calibration (MAPIE) on holdout — required by I-9 / E-S6-04
from agents.demand_prophet.training.conformal import calibrate_conformal
coverage_90 = calibrate_conformal(model, demand, alpha=0.10)
mlflow.log_metric(f'{CITY}_calibration_coverage_90', coverage_90)
assert coverage_90 >= 0.85, f'conformal coverage {coverage_90:.3f} < 0.85'

In [ ]:
# 8. Register model in MLflow Model Registry
model_name = f'mumbai_demand_prophet_hgt_tft' if CITY == 'mumbai' else 'demand_prophet_hgt_tft'
mlflow.pytorch.log_model(model, artifact_path='model', registered_model_name=model_name)
print(f'registered as {model_name}')

## Next steps

Promote the registered model from Dev → Staging → Production by following
`docs/runbooks/mlflow_promotion.md`. The promotion gate enforces:
- `convergence_speedup > 2.5` (transfer learning)
- `mumbai_calibration_coverage_90 >= 0.85` (conformal recalibration)
- 4-hour shadow evaluation
- 5% → 25% → 50% → 100% canary rollout